# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asiya-Akhtar/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [76]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("/content/flyrank-ml/data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Shape:", df.shape)

Rows: 30000
Columns: 44
Shape: (30000, 44)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

I will prioritize content using two current-state signals that were confirmed in the starter dataset.

1. **Staleness:** if `days_since_last_update >= 90`, add **2 points**.
2. **Low CTR despite search visibility:** if `impressions_90d >= 500`, `avg_position` is between 1 and 20, and `ctr < 0.5`, add **1 point**.

The total baseline score is the sum of these two signals.

The rule produces one reason code:

- `stale_and_low_ctr` — both signals fire
- `stale` — only the staleness signal fires
- `low_ctr_visible` — only the low-CTR visibility signal fires
- `no_flag` — neither signal fires

The corresponding action labels are:

- `refresh_and_fix_ctr`
- `refresh`
- `review_ctr`
- `monitor`

The rule is a simple decision-support baseline, not a prediction model.

In [77]:
# Section 1: define the two signals

df["stale"] = (
    df["days_since_last_update"] >= 90
)

df["low_ctr_visible"] = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"].between(1, 20))
    & (df["ctr"] < 0.5)
)

print("Stale counts:")
print(df["stale"].value_counts())

print("\nLow-CTR-visible counts:")
print(df["low_ctr_visible"].value_counts())

Stale counts:
stale
False    20655
True      9345
Name: count, dtype: int64

Low-CTR-visible counts:
low_ctr_visible
False    20255
True      9745
Name: count, dtype: int64


### Signal audit results

**Signal 1 — Staleness:** CONFIRMED.  
Observed declining rate was **60.85%** for stale content versus **51.20%** for non-stale content, a **9.64 percentage-point** difference.

**Signal 2 — Low CTR despite visibility:** CONFIRMED.  
Observed declining rate was **62.66%** for low-CTR visible content versus **50.14%** for the comparison group, a **12.52 percentage-point** difference.

Both signals are directional evidence for prioritization. The outcome field was used only to audit the signals, not as an input to the baseline score.

In [78]:
# Signal 1 bucket table

staleness_audit = (
    df.groupby("stale")
      .agg(
          n=("content_id", "size"),
          declining_rate=("trend_direction",
                          lambda x: (x == "down").mean() * 100)
      )
      .reset_index()
)

staleness_audit["bucket"] = staleness_audit["stale"].map({
    False: "Not stale (<90 days)",
    True: "Stale (90+ days)"
})

staleness_audit[
    ["bucket", "n", "declining_rate"]
]

,bucket,n,declining_rate
0,Not stale (<90 days),20655,51.203099
1,Stale (90+ days),9345,60.845372


In [79]:
# Signal 2 bucket table

ctr_audit = (
    df.groupby("low_ctr_visible")
      .agg(
          n=("content_id", "size"),
          declining_rate=("trend_direction",
                          lambda x: (x == "down").mean() * 100)
      )
      .reset_index()
)

ctr_audit["bucket"] = ctr_audit["low_ctr_visible"].map({
    False: "Not low-CTR visible",
    True: "Low-CTR visible"
})

ctr_audit[
    ["bucket", "n", "declining_rate"]
]

,bucket,n,declining_rate
0,Not low-CTR visible,20255,50.140706
1,Low-CTR visible,9745,62.657773


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Scoring rule

The baseline score is:

`baseline_score = 2 × stale + 1 × low_ctr_visible`

Therefore:

- Score **3** = both signals fire
- Score **2** = stale only
- Score **1** = low CTR visible only
- Score **0** = neither signal fires

When both signals fire, the item receives one combined reason code and one combined action.

In [81]:
# Section 2: calculate baseline score

df["baseline_score"] = (
    2 * df["stale"].astype(int)
    + 1 * df["low_ctr_visible"].astype(int)
)

print(df["baseline_score"].value_counts().sort_index())

baseline_score
0    14439
1     6216
2     5816
3     3529
Name: count, dtype: int64


In [80]:
# Assign exactly ONE reason code per content item

def get_reason_code(row):
    if row["stale"] and row["low_ctr_visible"]:
        return "stale_and_low_ctr"
    elif row["stale"]:
        return "stale"
    elif row["low_ctr_visible"]:
        return "low_ctr_visible"
    else:
        return "no_flag"

df["reason_code"] = df.apply(get_reason_code, axis=1)

print(df["reason_code"].value_counts())

reason_code
no_flag              14439
low_ctr_visible       6216
stale                 5816
stale_and_low_ctr     3529
Name: count, dtype: int64


In [82]:
# Assign exactly ONE action label

action_map = {
    "stale_and_low_ctr": "refresh_and_fix_ctr",
    "stale": "refresh",
    "low_ctr_visible": "review_ctr",
    "no_flag": "monitor"
}

df["action"] = df["reason_code"].map(action_map)

print(df["action"].value_counts())

action
monitor                14439
review_ctr              6216
refresh                 5816
refresh_and_fix_ctr     3529
Name: count, dtype: int64


In [85]:
from pathlib import Path

# Keep days_since_last_update temporarily so we can use it as a tie-breaker.
queue = (
    df[
        [
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "days_since_last_update"
        ]
    ]
    .sort_values(
        ["baseline_score", "days_since_last_update"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

# Remove the tie-breaker from the final CSV.
queue = queue[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
]

# Write the required CSV.
output_dir = Path("/content/flyrank-ml/work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

queue.to_csv(output_path, index=False)

print("Wrote:", output_path)
print("Rows:", len(queue))

display(queue.head(10))

Wrote: /content/flyrank-ml/work/outputs/baseline_action_score.csv
Rows: 30000


,content_id,baseline_score,reason_code,action
0,content_72496874f806,3,stale_and_low_ctr,refresh_and_fix_ctr
1,content_7f116ae1f6f5,3,stale_and_low_ctr,refresh_and_fix_ctr
2,content_fe16a55cd13d,3,stale_and_low_ctr,refresh_and_fix_ctr
3,content_cf56e2e2e282,3,stale_and_low_ctr,refresh_and_fix_ctr
4,content_77d4d5930e5e,3,stale_and_low_ctr,refresh_and_fix_ctr
5,content_c2d929d83eaa,3,stale_and_low_ctr,refresh_and_fix_ctr
6,content_928af3e22c80,3,stale_and_low_ctr,refresh_and_fix_ctr
7,content_0a91db491d14,3,stale_and_low_ctr,refresh_and_fix_ctr
8,content_6226ee6adc91,3,stale_and_low_ctr,refresh_and_fix_ctr
9,content_e3ff1b093148,3,stale_and_low_ctr,refresh_and_fix_ctr


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

# Get the top 20 for review

top20 = (
    df.sort_values(
        ["baseline_score", "days_since_last_update"],
        ascending=[False, False]
    )
    .head(20)
)

top20[
    [
        "content_id",
        "content_type",
        "days_since_last_update",
        "ctr",
        "avg_position",
        "impressions_90d",
        "baseline_score",
        "reason_code",
        "action"
    ]
]

In [86]:
# Get the top 20 for review

top20 = (
    df.sort_values(
        ["baseline_score", "days_since_last_update"],
        ascending=[False, False]
    )
    .head(20)
)

top20[
    [
        "content_id",
        "content_type",
        "days_since_last_update",
        "ctr",
        "avg_position",
        "impressions_90d",
        "baseline_score",
        "reason_code",
        "action"
    ]
]

,content_id,content_type,days_since_last_update,ctr,avg_position,impressions_90d,baseline_score,reason_code,action
7452,content_72496874f806,keyword article,301,0.24,5.8,821,3,stale_and_low_ctr,refresh_and_fix_ctr
26840,content_7f116ae1f6f5,keyword article,301,0.42,9.0,954,3,stale_and_low_ctr,refresh_and_fix_ctr
5327,content_fe16a55cd13d,keyword article,194,0.33,16.4,4556,3,stale_and_low_ctr,refresh_and_fix_ctr
16751,content_cf56e2e2e282,keyword article,194,0.15,19.7,61678,3,stale_and_low_ctr,refresh_and_fix_ctr
26799,content_77d4d5930e5e,keyword article,194,0.24,18.6,828,3,stale_and_low_ctr,refresh_and_fix_ctr
12045,content_c2d929d83eaa,keyword article,193,0.20,17.9,7558,3,stale_and_low_ctr,refresh_and_fix_ctr
20837,content_928af3e22c80,keyword article,193,0.12,15.8,1697,3,stale_and_low_ctr,refresh_and_fix_ctr
21268,content_0a91db491d14,keyword article,193,0.49,10.5,13299,3,stale_and_low_ctr,refresh_and_fix_ctr
11630,content_6226ee6adc91,keyword article,183,0.18,17.8,545,3,stale_and_low_ctr,refresh_and_fix_ctr
22872,content_e3ff1b093148,keyword article,183,0.28,7.8,1408,3,stale_and_low_ctr,refresh_and_fix_ctr


### Top-20 review

The top 20 are ranked by baseline score, with staler content first when scores tie.

The recommendations are decision-support only. A high score means the item matches the rule's observed signals; it does not prove that the recommended action will improve performance.

In [87]:
# Build the top-20 review table

review20 = top20[
    [
        "content_id",
        "days_since_last_update",
        "ctr",
        "avg_position",
        "impressions_90d",
        "baseline_score",
        "reason_code",
        "action"
    ]
].copy()

review20["confidence_note"] = (
    "Both confirmed signals fire; priority is supported by the baseline evidence."
)

review20["what_would_make_it_wrong"] = (
    "Low CTR may reflect search intent, SERP features, ranking limitations, "
    "or another factor that a refresh may not fix."
)

review20[
    [
        "content_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
7452,content_72496874f806,refresh_and_fix_ctr,stale_and_low_ctr,Both confirmed signals fire; priority is suppo...,"Low CTR may reflect search intent, SERP featur..."
26840,content_7f116ae1f6f5,refresh_and_fix_ctr,stale_and_low_ctr,Both confirmed signals fire; priority is suppo...,"Low CTR may reflect search intent, SERP featur..."
5327,content_fe16a55cd13d,refresh_and_fix_ctr,stale_and_low_ctr,Both confirmed signals fire; priority is suppo...,"Low CTR may reflect search intent, SERP featur..."
16751,content_cf56e2e2e282,refresh_and_fix_ctr,stale_and_low_ctr,Both confirmed signals fire; priority is suppo...,"Low CTR may reflect search intent, SERP featur..."
26799,content_77d4d5930e5e,refresh_and_fix_ctr,stale_and_low_ctr,Both confirmed signals fire; priority is suppo...,"Low CTR may reflect search intent, SERP featur..."
12045,content_c2d929d83eaa,refresh_and_fix_ctr,stale_and_low_ctr,Both confirmed signals fire; priority is suppo...,"Low CTR may reflect search intent, SERP featur..."
20837,content_928af3e22c80,refresh_and_fix_ctr,stale_and_low_ctr,Both confirmed signals fire; priority is suppo...,"Low CTR may reflect search intent, SERP featur..."
21268,content_0a91db491d14,refresh_and_fix_ctr,stale_and_low_ctr,Both confirmed signals fire; priority is suppo...,"Low CTR may reflect search intent, SERP featur..."
11630,content_6226ee6adc91,refresh_and_fix_ctr,stale_and_low_ctr,Both confirmed signals fire; priority is suppo...,"Low CTR may reflect search intent, SERP featur..."
22872,content_e3ff1b093148,refresh_and_fix_ctr,stale_and_low_ctr,Both confirmed signals fire; priority is suppo...,"Low CTR may reflect search intent, SERP featur..."


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

The weakest recommendations are borderline cases where the rule fires but the evidence may be less convincing.

Examples include items with CTR close to the 0.5% threshold, lower impression volume, or average positions toward the bottom of the 1–20 visibility range.

These cases should be treated as review candidates rather than guaranteed refresh opportunities.

In [88]:
# Identify borderline / weaker recommendations

weak_picks = df[
    (df["baseline_score"] > 0)
    & (
        (df["ctr"] >= 0.4)
        | (df["impressions_90d"] < 1000)
        | (df["avg_position"] > 15)
    )
].sort_values(
    ["baseline_score", "days_since_last_update"],
    ascending=[False, False]
).head(10)

weak_picks[
    [
        "content_id",
        "days_since_last_update",
        "ctr",
        "avg_position",
        "impressions_90d",
        "baseline_score",
        "reason_code",
        "action"
    ]
]

,content_id,days_since_last_update,ctr,avg_position,impressions_90d,baseline_score,reason_code,action
7452,content_72496874f806,301,0.24,5.8,821,3,stale_and_low_ctr,refresh_and_fix_ctr
26840,content_7f116ae1f6f5,301,0.42,9.0,954,3,stale_and_low_ctr,refresh_and_fix_ctr
5327,content_fe16a55cd13d,194,0.33,16.4,4556,3,stale_and_low_ctr,refresh_and_fix_ctr
16751,content_cf56e2e2e282,194,0.15,19.7,61678,3,stale_and_low_ctr,refresh_and_fix_ctr
26799,content_77d4d5930e5e,194,0.24,18.6,828,3,stale_and_low_ctr,refresh_and_fix_ctr
12045,content_c2d929d83eaa,193,0.20,17.9,7558,3,stale_and_low_ctr,refresh_and_fix_ctr
20837,content_928af3e22c80,193,0.12,15.8,1697,3,stale_and_low_ctr,refresh_and_fix_ctr
21268,content_0a91db491d14,193,0.49,10.5,13299,3,stale_and_low_ctr,refresh_and_fix_ctr
11630,content_6226ee6adc91,183,0.18,17.8,545,3,stale_and_low_ctr,refresh_and_fix_ctr
2016,content_67a766790dd2,106,0.18,14.9,569,3,stale_and_low_ctr,refresh_and_fix_ctr


### Leakage check

The baseline score uses only current-state fields:

- `days_since_last_update`
- `impressions_90d`
- `avg_position`
- `ctr`

I did not use `trend_pct` or `trend_direction` as score features. They were used only to audit whether the two proposed signals were directionally useful.

No future-window data is used in the baseline score.

In [89]:
# Explicit leakage check

score_features = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]

forbidden_features = [
    "trend_pct",
    "trend_direction",
    "is_declining_label"
]

print("Score features:")
for col in score_features:
    print(" -", col)

print("\nForbidden/outcome-derived fields:")
for col in forbidden_features:
    print(" -", col, "NOT USED IN SCORE")

assert "trend_pct" not in score_features
assert "trend_direction" not in score_features
assert "is_declining_label" not in score_features

print("\nLeakage check: PASSED")

Score features:
 - days_since_last_update
 - impressions_90d
 - avg_position
 - ctr

Forbidden/outcome-derived fields:
 - trend_pct NOT USED IN SCORE
 - trend_direction NOT USED IN SCORE
 - is_declining_label NOT USED IN SCORE

Leakage check: PASSED


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

# Self-check

- [x] Section 1 filled with rule, reason codes, and two signal audits.
- [x] Signal 1: staleness — CONFIRMED.
- [x] Signal 2: low CTR despite visibility — CONFIRMED.
- [x] Both bucket tables show `n`.
- [x] Section 2 contains the score, reason code, action, ranking, and CSV writing.
- [x] `baseline_action_score.csv` is generated by the notebook and is not intended for Git.
- [x] Section 3 contains the top-20 review.
- [x] Each reviewed item has action, reason code, confidence note, and what could make it wrong.
- [x] Section 4 identifies weak picks and performs a leakage check.
- [x] No `trend_pct` or `trend_direction` is used as a score feature.
- [x] No future-window information is used in the score.
- [x] Claims are phrased as observed/measured/directional decision-support evidence.
- [x] No client names, URLs, or private queries are included.
- [ ] Run the complete notebook top-to-bottom with no errors.
- [ ] Commit the executed notebook under `work/notebooks/w04_baseline_score.ipynb`.
- [ ] Submit the repository URL.